In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

# %%
spark = SparkSession.builder \
    .appName("SECOP_RegresionLineal") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/14 19:44:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/14 19:44:59 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [2]:
# Cargar datos
df = spark.read.parquet("/opt/spark-data/processed/secop_features.parquet")

# Renombrar columnas para consistencia
df = df.withColumnRenamed("valor_del_contrato_num", "label") \
       .withColumnRenamed("features_raw", "features")

In [3]:
df.show()

+-----------------------+-----------+--------------------+--------------------+-----------+--------------------+------------------+-------------------+---------------------+--------------------+---------------+----+-----+------------+---------------------+----+---+--------------------+----------------+----------+--------------------+----------------+-----------------+--------------------+
|referencia_del_contrato|nit_entidad|      nombre_entidad|        departamento|     ciudad|    tipo_de_contrato|valor_del_contrato|     fecha_de_firma|duraci_n_del_contrato|proveedor_adjudicado|estado_contrato|year|month|       label|fecha_de_firma_parsed|anio|mes|tipo_de_contrato_idx|departamento_idx|ciudad_idx|tipo_de_contrato_vec|departamento_vec|       ciudad_vec|            features|
+-----------------------+-----------+--------------------+--------------------+-----------+--------------------+------------------+-------------------+---------------------+--------------------+---------------+----+-

In [4]:
# Filtrar valores nulos
df = df.filter(col("label").isNotNull())
print(f"Registros: {df.count():,}")
print(f"Columnas: {len(df.columns)}")

Registros: 100,000
Columnas: 24


In [5]:
from pyspark.sql.functions import log1p, expm1

### Pregunta: ¿Qué proporción usarías para train vs test?

*Respuesta:*  
La proporción seleccionada es *B) 70/30 - Balance clásico*.

Esta proporción ofrece un equilibrio adecuado entre la cantidad de datos utilizados para entrenar el modelo y la cantidad de datos reservados para evaluar su desempeño. Permite entrenar un modelo robusto sin sacrificar la capacidad de validarlo correctamente con datos no vistos.

### Consideración: ¿Qué pasa si tienes 1 millón de registros vs 1000?

La proporción óptima depende del tamaño del dataset:

- *Con 1000 registros (dataset pequeño):*  
  Es recomendable usar proporciones como *70/30 o 80/20*, ya que se necesita una cantidad suficiente de datos para entrenar el modelo, pero también mantener un conjunto de prueba representativo.

- *Con 1 millón de registros (dataset grande):*  
  Incluso una proporción como *90/10* sería válida, ya que el 10% representaría 100,000 registros, una muestra más que suficiente para evaluar el modelo. En este caso, se prioriza maximizar la cantidad de datos para entrenamiento.

*Conclusión:*  
En este proyecto se utiliza *70/30* por ser un estándar ampliamente aceptado y adecuado al volumen de datos disponible, garantizando un buen balance entre entrenamiento y evaluación.

In [6]:
train_ratio = 0.7  # TODO: Ajusta según tu decisión
test_ratio = 0.3

train, test = df.randomSplit([train_ratio, test_ratio], seed=42)

print(f"Train: {train.count():,} registros ({train_ratio*100:.0f}%)")
print(f"Test: {test.count():,} registros ({test_ratio*100:.0f}%)")

Train: 70,007 registros (70%)


Test: 29,993 registros (30%)


¿Por qué es importante usar seed=42?

Respuesta:

Se utiliza seed=42 para garantizar la reproducibilidad del experimento. Al fijar la semilla, la partición de los datos en entrenamiento y prueba será siempre la misma en cada ejecución, permitiendo comparar resultados de forma consistente y replicable.

In [9]:
## RETO 2: Configurar el Modelo
lr = LinearRegression(
    featuresCol="features",
    labelCol="label",   # <- aquí el cambio
    maxIter=200,
    regParam=0.1,           # puedes empezar con 0.1
    elasticNetParam=0.0     # Ridge (L2)
)
print("✓ Modelo configurado")
print(f"  maxIter: {lr.getMaxIter()}")
print(f"  regParam: {lr.getRegParam()}")

✓ Modelo configurado
  maxIter: 200
  regParam: 0.1


In [10]:
## PASO 3: Entrenar el Modelo
print("Entrenando modelo de regresión lineal...")
lr_model = lr.fit(train)

print("✓ Modelo entrenado")
print(f"  Iteraciones completadas: {lr_model.summary.totalIterations}")
print(f"  RMSE (train): ${lr_model.summary.rootMeanSquaredError:,.2f}")
print(f"  R² (train): {lr_model.summary.r2:.4f}")

Entrenando modelo de regresión lineal...


26/02/14 19:48:40 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/02/14 19:48:40 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/02/14 19:48:40 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


✓ Modelo entrenado
  Iteraciones completadas: 0
  RMSE (train): $2,263,409,418.55
  R² (train): 0.0202


RETO 3: Interpretar R²
Respuesta
Opción correcta: B) El modelo explica 65% de la varianza en los datos.

El coeficiente de determinación R² mide qué proporción de la variabilidad de la variable objetivo puede ser explicada por el modelo.
Un valor de R² = 0.65 significa que el 65% de la variación en el valor real de los datos es explicada por las variables predictoras incluidas en el modelo, mientras que el 35% restante se debe a factores no modelados, ruido o relaciones no capturadas.

No significa que el modelo sea “65% preciso”, ya que R² no es una métrica de exactitud, sino de capacidad explicativa.

¿Es 0.65 un buen R²?
Depende de:

El dominio del problema:
En problemas sociales o económicos (como contratos públicos), un R² de 0.65 es considerado bueno, ya que los datos suelen ser muy ruidosos y complejos.

La naturaleza de los datos:
En datos financieros o reales, valores entre 0.3 y 0.7 ya son razonables.
En problemas físicos o de ingeniería, se esperan R² más altos (>0.9).

El objetivo del modelo:
Si el objetivo es exploratorio o de apoyo a decisiones, 0.65 es bastante aceptable.
Si es un sistema crítico de predicción, podría requerirse un R² mayor.

In [11]:
# %% [markdown]
# ## RETO 4: Análisis de Predicciones
from pyspark.sql.functions import abs as spark_abs

predictions = lr_model.transform(test)

# Error absoluto y error porcentual
predictions_with_error = predictions.withColumn(
    "absolute_error",
    spark_abs(col("prediction") - col("label"))
).withColumn(
    "error_pct",
    (spark_abs(col("prediction") - col("label")) / col("label")) * 100
)

print("\n=== 10 PEORES PREDICCIONES (MAYOR ERROR ABSOLUTO) ===")
predictions_with_error.orderBy(col("absolute_error").desc()) \
    .select("label", "prediction", "absolute_error", "error_pct",
            "departamento", "tipo_de_contrato", "estado_contrato") \
    .show(10, truncate=False)

print("\n=== EJEMPLOS CON ERROR > 100% ===")
predictions_with_error.filter(col("error_pct") > 100) \
    .select("label", "prediction", "absolute_error", "error_pct",
            "departamento", "tipo_de_contrato", "estado_contrato") \
    .show(10, truncate=False)

# Comentario (para el reto):
# Patrón típico: errores grandes suelen aparecer en contratos muy altos (outliers),
# o categorías con poca frecuencia (pocas observaciones por tipo/departamento).



=== 10 PEORES PREDICCIONES (MAYOR ERROR ABSOLUTO) ===


+----------------+--------------------+---------------------+-----------------+------------+------------------------------+---------------+
|label           |prediction          |absolute_error       |error_pct        |departamento|tipo_de_contrato              |estado_contrato|
+----------------+--------------------+---------------------+-----------------+------------+------------------------------+---------------+
|3.84542139555E11|2596223.479478121   |3.845395433315205E11 |99.9993248533223 |Bolívar     |Prestación de servicios       |Modificado     |
|1.38432454805E11|5.340137456692725E9 |1.3309231734830728E11|96.14242378045307|Atlántico   |Obra                          |En ejecución   |
|1.0596945342E11 |3.789026472677028E7 |1.0593156315527322E11|99.96424416329053|Risaralda   |Prestación de servicios       |Modificado     |
|9.728E10        |1.825031944574797E7 |9.726174968055424E10 |99.9812393920171 |Magdalena   |Prestación de servicios       |Modificado     |
|5.0670034387E10 |7.

+-----------+--------------------+--------------------+------------------+------------------+-----------------------+---------------+
|label      |prediction          |absolute_error      |error_pct         |departamento      |tipo_de_contrato       |estado_contrato|
+-----------+--------------------+--------------------+------------------+------------------+-----------------------+---------------+
|1.8E7      |-7.479761996441507E7|9.279761996441507E7 |515.5423331356393 |Huila             |Prestación de servicios|terminado      |
|5.16E7     |1.537620868945428E8 |1.0216208689454281E8|197.98854049330004|Norte de Santander|Prestación de servicios|Modificado     |
|4.4E7      |1.537620868945428E8 |1.0976208689454281E8|249.45928839668824|Norte de Santander|Prestación de servicios|En ejecución   |
|2.835E7    |1.537620868945428E8 |1.2541208689454281E8|442.3706768766943 |Norte de Santander|Prestación de servicios|Modificado     |
|5.7333333E7|1.537620868945428E8 |9.642875389454281E7 |168.189

Patrón observado:
- Los errores más altos se concentran en contratos atípicos o combinaciones poco frecuentes de departamento/tipo/estado (poca data para aprender)
- También se observan predicciones negativas en algunos casos, lo que incrementa mucho el error porcentual
- Esto sugiere que el modelo lineal baseline con pocas features no captura bien la escala del valor del contrato


In [12]:
# %% [markdown]
# ## PASO 5: Evaluación Formal

# %%
# Crear evaluadores para diferentes métricas
evaluator_rmse = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_mae = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="mae"
)

evaluator_r2 = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="r2"
)

# Calcular métricas
rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

*Análisis Train vs Test:*

R² Train = 0.0181 y R² Test = 0.0097 son ambos muy bajos y cercanos.
Esto sugiere UNDERFITTING (el modelo no captura la relación real) y no overfitting.
Probables causas: pocas features relevantes y target altamente sesgado (outliers).


In [13]:
# **Compara tus resultados**

# %%
print("\n=== COMPARACIÓN TRAIN VS TEST ===")
print(f"R² Train:  {lr_model.summary.r2:.4f}")
print(f"R² Test:   {r2:.4f}")
print(f"Diferencia: {abs(lr_model.summary.r2 - r2):.4f}")


=== COMPARACIÓN TRAIN VS TEST ===
R² Train:  0.0202
R² Test:   0.0096
Diferencia: 0.0106


¿Hay overfitting? No.
¿Hay underfitting? Sí.
Justificación:
- R² Train (0.0181) y R² Test (0.0097) son ambos muy bajos, lo que indica que el modelo
apenas explica la variabilidad del valor del contrato (modelo demasiado simple / poca señal en features).
- La diferencia entre train y test (0.0084) es pequeña, por lo que no hay evidencia de que el modelo esté “memorizando” el entrenamiento; más bien está fallando en capturar la relación subyacente.
- Esto es consistente con un target altamente disperso (outliers) y con features limitadas (mes + variables categóricas), lo que reduce la capacidad predictiva.


In [14]:
# ## RETO 6: Analizar Coeficientes
import numpy as np

coefficients = lr_model.coefficients
intercept = lr_model.intercept

coef_array = np.array(coefficients)
abs_coefs = np.abs(coef_array)
top_5_idx = np.argsort(abs_coefs)[-5:][::-1]

print(f"\nIntercept: {intercept:,.4f}")
print("\n=== TOP 5 COEFICIENTES (ABS) ===")
for rank, idx in enumerate(top_5_idx, start=1):
    print(f"{rank}. Feature_{idx}: coef = {coef_array[idx]:.6f}")


Intercept: 445,634,028.8519

=== TOP 5 COEFICIENTES (ABS) ===
1. Feature_541: coef = 16668121427.106209
2. Feature_490: coef = 6904833425.577578
3. Feature_16: coef = 6875430089.804641
4. Feature_7: coef = 4795996386.616627
5. Feature_487: coef = -4693378178.116310


26/02/14 20:13:04 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
26/02/14 20:13:04 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:291)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:981)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:165)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:263)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:170)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:115)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.proce

 ¿Qué significa un coeficiente positivo vs negativo?

Un coeficiente indica cómo cambia la predicción cuando esa feature aumenta (o se activa), manteniendo las demás constantes:

Coeficiente positivo (+): si la feature sube (o si una categoría One-Hot está en 1), la predicción del valor del contrato aumenta.

Coeficiente negativo (−): si la feature sube (o la categoría está en 1), la predicción disminuye.

En este caso con OneHotEncoder:

Una categoría con coeficiente positivo empuja el valor predicho por encima del baseline (intercepto).

Una categoría con coeficiente negativo lo empuja por debajo del baseline.

In [39]:
# %%
# Guardar modelo
model_path = "/opt/spark-data/processed/linear_regression_model"
lr_model.write().overwrite().save(model_path)
print(f"\n✓ Modelo guardado en: {model_path}")



✓ Modelo guardado en: /opt/spark-data/processed/linear_regression_model


In [40]:
# %%
# Guardar predicciones
predictions_path = "/opt/spark-data/processed/predictions_lr.parquet"
predictions.write.mode("overwrite").parquet(predictions_path)
print(f"✓ Predicciones guardadas en: {predictions_path}")


26/02/14 02:41:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


✓ Predicciones guardadas en: /opt/spark-data/processed/predictions_lr.parquet


In [41]:
pred_real = predictions.withColumn("pred_real", expm1(col("prediction"))) \
                       .withColumn("label_real", expm1(col("label_log")))
pred_real.select("label_real", "pred_real").show(10, truncate=False)


+--------------------+--------------------+
|label_real          |pred_real           |
+--------------------+--------------------+
|1.7999999999999985E7|1.961713982460047E7 |
|1.2815399699999982E8|4.3441457447616935E7|
|2.234400000000001E7 |3.5790728547084495E7|
|5.160000000000008E7 |2.8905633293772243E7|
|4.399999999999999E7 |2.0128887023003608E7|
|2.835000000000005E7 |2.8905633293772243E7|
|5.7333332999999925E7|2.8905633293772243E7|
|5.7333332999999925E7|2.8905633293772243E7|
|4.720000000000002E7 |1.9235210097088557E7|
|3.6000000000000045E7|2.0128887023003608E7|
+--------------------+--------------------+
only showing top 10 rows



In [42]:
# %%
print("\n" + "="*60)
print("RESUMEN REGRESIÓN LINEAL")
print("="*60)
print(f"✓ Modelo entrenado con {train.count():,} registros")
print(f"✓ Evaluado con {test.count():,} registros")
print(f"✓ RMSE: ${rmse:,.2f}")
print(f"✓ R²: {r2:.4f}")
print(f"✓ Próximo paso: Probar regularización (notebook 07)")
print("="*60)


RESUMEN REGRESIÓN LINEAL


✓ Modelo entrenado con 70,007 registros
✓ Evaluado con 29,993 registros
✓ RMSE: $1.24
✓ R²: 0.2258
✓ Próximo paso: Probar regularización (notebook 07)


In [22]:
# %%
spark.stop()